# A multiplier-free CNN on MNIST

A tropical 2-D convolution **is** a learned grayscale *morphological* operation:
`y = max_{ci,kh,kw} (weight + patch)` (dilation) or `min` (erosion). No multiplies
— a structuring element is added to each image patch and reduced by a compare.
This notebook builds a CNN entirely from such ops (conv → tropical-norm →
power-of-2 leaky-relu → max-pool → compositional tropical head), verifies it is
multiplier-free, and trains it on MNIST with a **sign-only optimizer**.


In [ ]:
# --- Setup: install TikiTorch (run once) ---
# TikiTorch is research code, not on PyPI. In Colab, install from the repo.
try:
    import tikitorch  # already installed?
except ImportError:
    REPO_URL = "https://github.com/nyuolab/tikitorch.git"
    import subprocess, sys, os
    if not os.path.exists("tikitorch_repo"):
        subprocess.run(["git", "clone", REPO_URL, "tikitorch_repo"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "tikitorch_repo"], check=False)
    import tikitorch
import torch
print("TikiTorch", getattr(tikitorch, "__version__", "?"), "| torch", torch.__version__)


## See the morphology: TropicalConv2d as dilation / erosion


In [ ]:
import torch, tikitorch.nn as nn
img = torch.zeros(1, 1, 16, 16); img[0, 0, 6:10, 6:10] = 1.0   # a bright square

dil = nn.TropicalConv2d(1, 1, 3, padding=1, mode="dilation", bias=False)
ero = nn.TropicalConv2d(1, 1, 3, padding=1, mode="erosion",  bias=False)
with torch.no_grad():
    dil.weight.zero_(); ero.weight.zero_()           # flat 3x3 structuring element
    d = dil(img)[0, 0]; e = ero(img)[0, 0]
print("input bright pixels :", int((img > 0.5).sum()))
print("after dilation      :", int((d > 0.5).sum()), "(square grew — max-plus)")
print("after erosion       :", int((e > 0.5).sum()), "(square shrank — min-plus)")


## The model: a multiplier-free morphological feature extractor

`TropicalCNN` alternates dilation/erosion conv blocks (opening/closing structure,
much more expressive than all-dilation), pools with `max`, normalizes with a
power-of-2 `TropicalLayerNorm`, keeps coarse spatial layout with an adaptive
max-pool grid, and ends in a head. Two honest choices for the head:

- **`head_type="linear"`** — a standard `nn.Linear` classifier. The *feature
  extractor* is fully multiplier-free; the head is a single **declared DSP
  boundary** (`assert_closed(model, allow={"head"})`). This is the strong,
  honest configuration: ~**96–97% MNIST with a sign-only optimizer**.
- **`head_type="tropical"`** — a compositional dilation∘erosion head, so the
  **entire** model is multiplier-free (`assert_closed(model)` passes outright).
  This costs accuracy at the 10-way head — the *closure penalty* made visible.

We train the strong configuration first, then quantify the fully-closed cost.


In [ ]:
import tikitorch as tt
from tikitorch.audit import assert_closed, find_closure_violations

model = tt.models.TropicalCNN(in_channels=1, num_classes=10, channels=(16, 32),
                              pool_grid=4, head_type="linear",
                              use_soft=True, tau=1.0)
model.loss_type = "cross_entropy"
# The feature extractor is multiplier-free; only the head is a declared boundary:
print("closure violations:", find_closure_violations(model))   # -> [('head', 'Linear')]
assert_closed(model, allow={"head"})
print("Backbone is multiplier-free ✓ (head = declared DSP boundary). params:", model.param_count)


## Load MNIST and train with a sign-only optimizer + soft→hard anneal


In [ ]:
import torch
from torchvision import datasets, transforms
import tikitorch.optim as optim

tfm = transforms.ToTensor()
train = datasets.MNIST(root="./data", train=True,  download=True, transform=tfm)
test  = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)
train_loader = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test,  batch_size=512)

opt = optim.SignSGD(model.parameters(), lr=2e-3, momentum=0.9)
EPOCHS = 3
total = EPOCHS * len(train_loader)
step = 0
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in train_loader:
        model.set_tau(max(1.0 * (1 - step / (0.8 * total)), 0.0))   # anneal tau->0
        opt.zero_grad()
        model.compute_loss(xb, yb).backward(); opt.step()
        step += 1
    model.set_tau(0.0); model.eval()
    correct = tot = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            correct += (model(xb).argmax(-1) == yb).sum().item(); tot += yb.numel()
    print(f"epoch {epoch}: tau->0  test_acc = {correct/tot:.4f}   "
          f"(multiplier-free backbone + SignSGD; head is the one declared boundary)")


## The fully-closed model, and the closure penalty

Now swap to the **tropical head** so the *entire* network is multiplier-free —
`assert_closed(model)` passes with no allowances. On MNIST this 10-way tropical
head is the bottleneck (the single-argmax routing is weaker than a dense head),
so accuracy drops — that gap *is* the closure penalty at the head, and reporting
both numbers is the honest comparison. The morphological **features** are
identical and excellent (they drove the ~96% above); only the head changed.


In [ ]:
closed = tt.models.TropicalCNN(in_channels=1, num_classes=10, channels=(16, 32),
                               pool_grid=2, head_type="tropical",
                               use_soft=True, tau=1.0)
closed.loss_type = "tropical_ce"
assert_closed(closed)     # no allow=... needed: the WHOLE model is multiplier-free
print("Fully multiplier-free CNN ✓ (forward + backward + optimizer, zero DSP).")

opt = optim.SignSGD(closed.parameters(), lr=2e-3, momentum=0.9)
total = 2 * len(train_loader); step = 0
for epoch in range(2):
    closed.train()
    for xb, yb in train_loader:
        closed.set_tau(max(1.0 * (1 - step / (0.8 * total)), 0.0))
        opt.zero_grad(); closed.compute_loss(xb, yb).backward(); opt.step(); step += 1
    closed.set_tau(0.0); closed.eval()
    c = t = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            c += (closed(xb).argmax(-1) == yb).sum().item(); t += yb.numel()
    print(f"epoch {epoch}: FULLY-closed test_acc = {c/t:.4f}  (vs ~0.96 with a boundary head)")

print("\\nThe deploy graph (tau=0) is exactly what maps to the zero-DSP FPGA kernel.")
